# 10 — Train DINOv2-S baseline (one fold smoke → 5-fold)

**Attach:** competition data (optional if cache-only), `rsna-knee-cache-v1`, code repo dataset, DINOv2 weights dataset.

Accelerator: GPU. Internet: ON for first torch.hub fallback; prefer offline weights.

In [ ]:
from pathlib import Path
import sys, subprocess

REPO = next(p for p in [
    Path('/kaggle/input/rsna-knee-code'),
    Path('/kaggle/input/rsna-knee-abnormality-detection-model'),
] if (p / 'src' / 'rsna_knee').exists())
sys.path.insert(0, str(REPO / 'src'))

CACHE = next(p for p in [
    Path('/kaggle/input/rsna-knee-cache-v1'),
    Path('/kaggle/input/rsna-knee-cache-v1/cache_v1'),
    Path('/kaggle/working/cache_v1'),
] if p.exists() and any(p.glob('*.npz')))

DATA = next((p for p in [
    Path('/kaggle/input/rsna-knee-abnormality-detection'),
    Path('/kaggle/input/rsna-knee-abnormalities-detection'),
] if (p / 'train.csv').exists()), None)

WEIGHTS = None
for p in Path('/kaggle/input').glob('**/*dinov2*vits14*.pth'):
    WEIGHTS = p; break
for p in Path('/kaggle/input').glob('**/dinov2_vits14_pretrain.pth'):
    WEIGHTS = p; break

FOLDS = REPO / 'data' / 'folds' / 'folds_v1.csv'
TRAIN_CSV = (DATA / 'train.csv') if DATA else (REPO / 'data' / 'raw' / 'train.csv')
WEAK = REPO / 'data' / 'processed' / 'weak_labels_v1.csv'
OUT = Path('/kaggle/working/baseline_dinov2_s')
OUT.mkdir(exist_ok=True)
print('REPO', REPO)
print('CACHE', CACHE, 'n', len(list(CACHE.glob('*.npz'))))
print('WEIGHTS', WEIGHTS)
print('TRAIN_CSV', TRAIN_CSV)

In [ ]:
%pip -q install pyyaml scikit-learn tqdm

FOLD = 0
EPOCHS = 3  # smoke; raise to 5 for real
cmd = [
    sys.executable, str(REPO / 'scripts' / 'train_baseline_fold.py'),
    '--config', str(REPO / 'configs' / 'baseline_dinov2_s.yaml'),
    '--train-csv', str(TRAIN_CSV),
    '--folds', str(FOLDS),
    '--cache-dir', str(CACHE),
    '--fold', str(FOLD),
    '--epochs', str(EPOCHS),
    '--out-dir', str(OUT),
]
if WEIGHTS:
    cmd += ['--weights', str(WEIGHTS)]
if WEAK.exists():
    cmd += ['--weak-csv', str(WEAK)]
print(' '.join(cmd))
subprocess.check_call(cmd)

Log the printed `val_macro_auc` into `docs/experiments.md` on the Mac after download.
Then loop folds 0–4 and blend OOF for a first public submit.